# <font color='black'>Регрессионный анализ: панельные данные и каузальность, 2026 </font>
## <font color='black'> Метод инструментальных переменных </font>

Рассмотрим пример применения метода инструментальных переменных из статьи Hansford T. G., Gomez B. T. (2010) Estimating the Electoral Effects of Voter Turnout.The American Political Science Review, Vol. 104, No. 2, pp. 268 - 288. (статью можете скачать по ссылке https://disk.yandex.ru/i/kIc9Ed09_vaX8w).

Ниже представлено краткое описание данных:

* Year - Election Year
* FIPS_County - FIPS County Code
* DemVoteShare2 - Democratic President Candidate’s Vote Share
* Turnout - turnout (in percentage)   
* DNormPrcp_KRIG - Election day rainfall differenced from normal rain for the day

Откроем массив и оставим только необходимые для анализа переменные:

In [ ]:
import pandas as pd
import numpy as np
import numpy.linalg as la
import statsmodels.formula.api as smf
!pip install linearmodels
from linearmodels import IV2SLS

In [ ]:
IVdata = pd.read_csv('IVdata.csv')
IVdata_short = IVdata[['Year', 'FIPS_County', 'Turnout', 'DemVoteShare2', 'DNormPrcp_KRIG']].dropna()
IVdata_short.head()

,Year,FIPS_County,Turnout,DemVoteShare2,DNormPrcp_KRIG
0,1948,4021,27.17562,61.54376,-0.025283
1,1948,4019,44.74918,51.04443,-0.010943
2,1948,4009,51.46835,63.88889,-0.035849
3,1948,4025,55.56316,50.87096,-0.044717
4,1948,4013,42.93417,52.53817,-0.025660


Оценим предварительную модель, не применяя метод IV. Какие источники эндогенности присутствуют в данном случае? Можно ли доверять полученным результатам?

In [ ]:
m1 = smf.ols(formula='DemVoteShare2 ~ Turnout + C(Year)', data=IVdata_short).fit(cov_type='cluster', cov_kwds={'groups': IVdata_short['FIPS_County']})
print(m1.summary())

                            OLS Regression Results                            
Dep. Variable:          DemVoteShare2   R-squared:                       0.281
Model:                            OLS   Adj. R-squared:                  0.280
Method:                 Least Squares   F-statistic:                     5416.
Date:                Sun, 10 May 2026   Prob (F-statistic):               0.00
Time:                        22:06:54   Log-Likelihood:            -1.0339e+05
No. Observations:               27401   AIC:                         2.068e+05
Df Residuals:                   27386   BIC:                         2.069e+05
Df Model:                          14                                         
Covariance Type:              cluster                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          59.0855      1.240     

Сначала попробуем реализовать метод IV последовательно. Оценим модель первого шага:

In [ ]:
stage1 = smf.ols('Turnout ~ DNormPrcp_KRIG + C(Year)', data=IVdata_short).fit(cov_type='cluster', cov_kwds={'groups': IVdata_short['FIPS_County']})
print(stage1.summary())

                            OLS Regression Results                            
Dep. Variable:                Turnout   R-squared:                       0.226
Model:                            OLS   Adj. R-squared:                  0.226
Method:                 Least Squares   F-statistic:                     1526.
Date:                Sun, 10 May 2026   Prob (F-statistic):               0.00
Time:                        22:06:54   Log-Likelihood:                -99833.
No. Observations:               27401   AIC:                         1.997e+05
Df Residuals:                   27386   BIC:                         1.998e+05
Df Model:                          14                                         
Covariance Type:              cluster                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          61.6937      0.217    2

Выведем частную F-статистику для инструмента для того, чтобы проверить гипотезу о силе инструмента. Для того, чтобы можно было утверждать, что инструмент не является слабым, F-статистика должна превышать 10

In [ ]:
stage1_short = smf.ols('Turnout ~ C(Year)', data=IVdata_short).fit(cov_type='cluster', cov_kwds={'groups': IVdata_short['FIPS_County']})

ssr_full = stage1.ssr
ssr_restricted = stage1_short.ssr
dif = 1
df_full = stage1.df_resid

f_statistic = ((ssr_restricted - ssr_full) / dif) / (ssr_full / df_full)
print(f"Частная F-статистика: {f_statistic:.2f}")

Частная F-статистика: 51.02


Либо используем тест Вальда (Wald test):

In [ ]:
wald_result = stage1.wald_test('DNormPrcp_KRIG = 0')
print(f"Статистика теста Вальда: {wald_result.statistic[0][0]:.2f}, p-value: {wald_result.pvalue:.4f}")

Статистика теста Вальда: 53.91, p-value: 0.0000


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(


Реализуем второй шаг метода инструментальных переменных, предварительно сохранив предсказанные значения из первой модели (stage1)

In [ ]:
IVdata_short['fitted'] = stage1.predict()

stage2 = smf.ols('DemVoteShare2 ~ fitted + C(Year)', data=IVdata_short).fit(cov_type='cluster', cov_kwds={'groups': IVdata_short['FIPS_County']})
print(stage2.summary())

                            OLS Regression Results                            
Dep. Variable:          DemVoteShare2   R-squared:                       0.267
Model:                            OLS   Adj. R-squared:                  0.267
Method:                 Least Squares   F-statistic:                     5679.
Date:                Sun, 10 May 2026   Prob (F-statistic):               0.00
Time:                        22:06:55   Log-Likelihood:            -1.0364e+05
No. Observations:               27401   AIC:                         2.073e+05
Df Residuals:                   27386   BIC:                         2.074e+05
Df Model:                          14                                         
Covariance Type:              cluster                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          26.9099     10.950     

Реализуем метод IV автоматически:

In [ ]:
iv_model = IV2SLS.from_formula('DemVoteShare2 ~ 1 + C(Year) + [Turnout ~ DNormPrcp_KRIG]', data=IVdata_short).fit(cov_type='clustered', clusters=IVdata_short['FIPS_County'])
print(iv_model)

                          IV-2SLS Estimation Summary                          
Dep. Variable:          DemVoteShare2   R-squared:                      0.1302
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1297
No. Observations:               27401   F-statistic:                 7.564e+04
Date:                Sun, May 10 2026   P-value (F-stat)                0.0000
Time:                        22:06:56   Distribution:                 chi2(14)
Cov. Estimator:             clustered                                         
                                                                              
                                Parameter Estimates                                
                 Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
-----------------------------------------------------------------------------------
Intercept           26.910     11.587     2.3225     0.0202      4.2005      49.619
C(Year)[T.1952]    -15.832     2

Проверим сопоставимость оценок OLS и TSLS при помощи теста Ву-Хаусмана:

In [ ]:
print(iv_model.wu_hausman())

Wu-Hausman test of exogeneity
H0: All endogenous variables are exogenous
Statistic: 10.6776
P-value: 0.0011
Distributed: F(1,27385)


Результаты теста Саргана в данном случае недоступны, как мы видим. Причина этому - точная идентификация (количество инструментов = количество эндогенных предикторов)

In [ ]:
if hasattr(iv_model, 'sargan'):
    print(f"Статистика теста Хансена: {iv_model.sargan.stat:.3f}")
    print(f"p-value: {iv_model.sargan.pval:.3f}")

Статистика теста Хансена: nan
p-value: nan


## Практикум: часть 2

Откроем массив IV_data2.Rda

Ниже представлено краткое описание данных для этого классического примера из эконометрики оценивания эффекта образования на заработную плату посредством инструментальных переменных

* wage - outcome variable - natural logarithm (wage)
* education - years of education
* gender - male/female
* fcollege - Father's education: "yes" stands for higher education
* mcollege - Mother's education: "yes" stands for higher education
* ethnicity - Afro-American / Latin American / other
* urban (dummy-variable: 1 - urban, 0 - rural)
* distance - distance from college

In [ ]:
!pip install pyreadr
import pyreadr as pyreadr

In [ ]:
dta2 = pyreadr.read_r('IV_data2.Rda')
dta2 = dta2[None]
dta2

,gender,ethnicity,fcollege,mcollege,urban,wage,distance,education
rownames,,,,,,,,
1,male,other,yes,no,1.0,8.09,0.2,12.0
2,female,other,no,no,1.0,8.09,0.2,12.0
3,male,other,no,no,1.0,8.09,0.2,12.0
4,male,afam,no,no,1.0,8.09,0.2,12.0
5,female,other,no,no,1.0,8.09,0.4,13.0
...,...,...,...,...,...,...,...,...
9391,male,afam,no,no,0.0,8.89,0.8,13.0
9401,male,afam,no,no,0.0,8.89,0.8,15.0
9411,male,other,no,no,0.0,8.89,0.8,12.0


Приведите необходимые преобразования переменных при необходимости и оцените при помощи 2МНК модель wage на education. Остальные переменные разделите на две подгруппы: контрольные переменные и инструментальные переменные. Выберите несколько инструментальных переменных, чтобы создалась ситуация чрезмерной идентификации.

Проинтерпретируйте полученные результаты, сравните с оценками соответствующей модели без инструментальных переменных.

Реализуйте все необходимые диагностики для модели с инструментальными переменными (тест на слабость инструментов, тест Ву-Хаусмана и тест Хансена) и проинтерпретируйте полученные результаты